# Clean messy data with datacom

[datacom](https://github.com/moodykhalif23/datacom) is a small, fast, open-source cleaner for data files too big or too messy for a spreadsheet. It fixes numbers, dates, yes/no values and phone numbers, drops empty and duplicate rows, masks personal data, and prepares document text for AI training, reporting every change it makes.

This notebook installs it from PyPI, makes a messy sales export, and cleans it. Then it prepares a small text corpus the way training data is prepared.

In [ ]:
!pip install -q datacom

## A messy export

Fifty thousand invoices with the usual problems: amounts written five ways, mixed date formats, inconsistent casing, stray spaces in names, yes/no in several spellings, and some rows exported twice.

In [ ]:
import csv
import random

random.seed(7)
customers = ["Acme Ltd", "acme ltd", "Kenya Power", "Safaricom PLC", " Tullow Oil", "Kenya Airways"]
regions = ["Nairobi", "nairobi", "Mombasa ", "Kisumu", "NAIROBI"]
amounts = ["KES 1,250.00", "3400", "(500)", "1,200.50", "12,000", "", "N/A"]
dates = ["2025-01-15", "15/02/2025", "2025/04/20", "2025-03-01"]
paid = ["Yes", "no", "Y", "N", "TRUE"]

with open("sales.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["Invoice No", "Customer Name ", "Region", "Amount (KES)", "Invoice Date", "Paid?", "Email", "Phone"])
    for i in range(50_000):
        row = [1000 + i, random.choice(customers), random.choice(regions), random.choice(amounts),
               random.choice(dates), random.choice(paid), f"buyer{i}@example.co.ke",
               f"07{random.randint(10, 99)} {random.randint(100, 999)} {random.randint(100, 999)}"]
        w.writerow(row)
        if i % 100 == 0:
            w.writerow(row)

## What is in it?

`profile` reads the whole file and lists each column's type, missing values, common values and the problems worth fixing.

In [ ]:
import datacom

report = datacom.profile("sales.csv")
print(f"{report['rows']:,} rows")
for issue in report["issues"]:
    print("-", issue)

## Clean it

`clean` returns the run record: the SHA-256 of the input and output, and every rule applied with what it changed.

In [ ]:
record = datacom.clean("sales.csv", "clean.parquet", snake=True, dedupe=True, fold_case=True, phone=254, redact="all")
print(f"{record['rows_in']:,} rows in, {record['rows_out']:,} out")
for rule in record["rules"]:
    print(rule)

In [ ]:
import pandas as pd

pd.read_parquet("clean.parquet").head(10)

## Text for AI training

A small corpus with HTML left in, garbled characters, near-duplicates, documents too short to be useful, and one that copies a question from an evaluation set. `text` cleans the document column, and the other options drop rows, each counted on its own.

In [ ]:
import json

words = ("the market opens at dawn and traders from the coast bring fish fruit cloth and spices "
         "to sell before the heat of the day while buyers walk between the stalls").split()


def doc(seed, n=80):
    r = random.Random(seed)
    return " ".join(r.choice(words) for _ in range(n)).capitalize() + "."


rows = []
for i in range(3000):
    body = doc(i)
    if i % 50 == 1:
        body = "<p>" + body + "</p><br>"
    if i % 50 == 2:
        body = body.replace("the", "café", 2).encode().decode("latin-1")
    if i % 50 == 3:
        body = "Too short to keep."
    rows.append({"id": i, "body": body})
    if i % 40 == 0:
        rows.append({"id": f"{i}-copy", "body": body.replace("dawn", "sunrise", 1)})

with open("corpus.jsonl", "w") as f:
    for r in rows:
        f.write(json.dumps(r) + "\n")
with open("evals.jsonl", "w") as f:
    f.write(json.dumps({"question": " ".join(doc(7).split()[10:30])}) + "\n")

record = datacom.clean("corpus.jsonl", "corpus-clean.jsonl", text=["body"], quality="default",
                       near_dupes=0.8, decontam="evals.jsonl")
print(f"{record['rows_in']:,} documents in, {record['rows_out']:,} kept")
for rule in record["rules"]:
    print(rule)

## More

- `datacom.query` filters, joins, derives and totals; `datacom.convert` changes format.
- Sources can be globs, `s3://`, `gs://` and `hf://datasets/...`, and an output folder gets one part per source.
- The same engine is a single binary for the terminal, and `datacom ui` opens it in a browser.

Source, documentation and releases: https://github.com/moodykhalif23/datacom